# https://data.cityofnewyork.us/Environment/2015-Street-Tree-Census-Tree-Data/uvpi-gqnh/about_data


In [1]:
# Import necessary libraries for data processing and spatial operations.
import pandas as pd  # For handling tabular data (CSV files).
import numpy as np  # For numerical operations and array manipulations.
from scipy.spatial import cKDTree  # For efficient spatial queries using KD-tree.
import scipy  # Import scipy to retrieve its version.
import tqdm  # Import the tqdm module for accessing its version.
from tqdm import tqdm as tqdm_progress  # Alias the tqdm function for progress bars.

# Debug: Confirm that imports are successful.
print("Debug: Libraries imported successfully.")

# Print the version of each imported library.
print(f"Debug: pandas version: {pd.__version__}")
print(f"Debug: numpy version: {np.__version__}")
print(f"Debug: scipy version: {scipy.__version__}")
print(f"Debug: tqdm version: {tqdm.__version__}")


Debug: Libraries imported successfully.
Debug: pandas version: 2.2.3
Debug: numpy version: 1.26.4
Debug: scipy version: 1.13.1
Debug: tqdm version: 4.67.1


In [2]:
# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for input datasets.
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"
tree_file = f"{base_dir}/2015_Street_Tree_Census_-_Tree_Data_20250221.csv" 
# https://data.cityofnewyork.us/Environment/2015-Street-Tree-Census-Tree-Data/uvpi-gqnh/about_data

# Define output file paths for enriched datasets.
output_train_csv = f"{sub_dir}/training_data_TREE_width_count.csv"
output_valid_csv = f"{sub_dir}/validation_data_TREE_width_count.csv"

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Tree data file path: {tree_file}")
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Tree data file path: /kaggle/input/eyds-base-dataset/2015_Street_Tree_Census_-_Tree_Data_20250221.csv
Debug: Output training CSV path: /kaggle/working//training_data_TREE_width_count.csv
Debug: Output validation CSV path: /kaggle/working//validation_data_TREE_width_count.csv


In [3]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the Haversine distance between two points in meters.
    Note: This function is not used in the vectorized KD-tree approach below but is included for reference.
    
    Parameters:
        lat1 (float): Latitude of the first point in degrees.
        lon1 (float): Longitude of the first point in degrees.
        lat2 (float): Latitude of the second point in degrees.
        lon2 (float): Longitude of the second point in degrees.
    
    Returns:
        float: Distance between the two points in meters.
    """
    R = 6371000  # Earth's radius in meters.
    # Convert decimal degrees to radians.
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    # Haversine formula to calculate distance.
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [4]:
def calculate_tree_features_vectorized(locations_df, tree_data, radii_meters):
    """
    Calculate tree features for each location within specified radii using a vectorized KD-tree approach.
    Computes the count of trees and average tree diameter within each radius for each location.
    
    Parameters:
        locations_df (pd.DataFrame): DataFrame containing at least 'Latitude' and 'Longitude' columns.
        tree_data (pd.DataFrame): DataFrame with tree details, including 'latitude', 'longitude', 'tree_dbh'.
        radii_meters (list): List of radii (in meters) for which to compute the features.
    
    Returns:
        pd.DataFrame: locations_df with additional columns 'tree_count_{radius}m' and 'tree_avg_diam_{radius}m' for each radius.
    """
    # Debug: Print the shapes of input DataFrames.
    print(f"Debug: Locations DataFrame shape: {locations_df.shape}")
    print(f"Debug: Tree DataFrame shape: {tree_data.shape}")

    # Get location coordinates (in degrees) and convert to numpy array.
    loc_coords = locations_df[['Latitude', 'Longitude']].values

    # Debug: Print the number of locations being processed.
    print(f"Debug: Number of locations to process: {len(loc_coords)}")

    # Get tree coordinates and convert to radians for the KD-tree.
    tree_coords = tree_data[['latitude', 'longitude']].values
    tree_coords_rad = np.radians(tree_coords)
    tree_kdtree = cKDTree(tree_coords_rad)

    # Debug: Confirm that the KD-tree was created.
    print("Debug: KD-tree for trees created successfully.")

    # Prepare dictionaries to store counts and average diameters for each radius.
    result_counts = {radius: np.zeros(len(locations_df), dtype=int) for radius in radii_meters}
    result_avg_diam = {radius: np.full(len(locations_df), np.nan) for radius in radii_meters}

    # Precompute each radius in radians (Earth's radius = 6371000 meters).
    radius_radians = {radius: radius / 6371000.0 for radius in radii_meters}

    # Debug: Print the radii being used.
    print(f"Debug: Radii (in meters) for feature computation: {radii_meters}")

    # Use only the tree_dbh for diameter (since tree data is already filtered for Alive trees).
    tree_data['tree_dbh'] = pd.to_numeric(tree_data['tree_dbh'], errors='coerce')
    tree_data['diameter'] = tree_data['tree_dbh']
    tree_diameters = tree_data['diameter'].values

    # Debug: Print a summary of tree diameters.
    print(f"Debug: Number of valid tree diameters: {tree_diameters[~np.isnan(tree_diameters)].size}")
    print(f"Debug: Sample tree diameters: {tree_diameters[:5]}")

    # Process locations in batches for efficiency.
    batch_size = 1000
    num_batches = int(np.ceil(len(locations_df) / batch_size))

    # Debug: Print the number of batches to process.
    print(f"Debug: Number of batches to process: {num_batches}")

    for i in tqdm_progress(range(num_batches), desc="Processing locations"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(locations_df))
        batch_coords = loc_coords[start_idx:end_idx]
        batch_coords_rad = np.radians(batch_coords)

        # For each radius, query the KD-tree.
        for radius in radii_meters:
            r_rad = radius_radians[radius]
            # Get list of indices (for each location in the batch) of trees within the radius.
            indices_list = tree_kdtree.query_ball_point(batch_coords_rad, r=r_rad, workers=-1)
            # Loop through each location in the batch and compute features.
            for j, indices in enumerate(indices_list):
                count = len(indices)
                result_counts[radius][start_idx + j] = count
                if count > 0:
                    # Get the diameters for the trees within this radius.
                    diameters = tree_diameters[indices]
                    valid_diameters = diameters[~np.isnan(diameters)]
                    if len(valid_diameters) > 0:
                        avg_diam = np.nanmean(valid_diameters)
                    else:
                        avg_diam = np.nan
                    result_avg_diam[radius][start_idx + j] = avg_diam

    # Append new columns to the locations DataFrame.
    for radius in radii_meters:
        col_count = f'tree_count_{radius}m'
        col_avg = f'tree_avg_diam_{radius}m'
        locations_df[col_count] = result_counts[radius]
        locations_df[col_avg] = result_avg_diam[radius]

    # Debug: Print the new columns added to the DataFrame.
    new_cols = [col for col in locations_df.columns if col.startswith('tree_')]
    print(f"Debug: New columns added to locations DataFrame: {new_cols}")

    return locations_df

In [5]:
def main():
    """
    Main function to calculate tree features (counts and average diameters) for training and validation datasets
    and save the augmented data to new CSV files.
    """
    # Define the radii (in meters) for which to calculate tree features.
    radii = [50, 100, 200, 300, 500, 750, 1000]

    # Debug: Print the radii being used.
    print(f"Debug: Radii for tree features: {radii}")

    # Read the tree data.
    print("Reading tree data...")
    try:
        tree_data = pd.read_csv(tree_file)
    except FileNotFoundError:
        print(f"Error: Tree data file not found at {tree_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the tree data.
    print(f"Debug: Tree DataFrame shape: {tree_data.shape}")
    print(f"Debug: Tree DataFrame columns: {tree_data.columns.tolist()}")

    # Filter for good trees (status 'Alive').
    tree_data = tree_data[tree_data['status'] == 'Alive']

    # Debug: Print the shape after filtering for alive trees.
    print(f"Debug: Tree DataFrame shape after filtering for 'Alive' trees: {tree_data.shape}")

    # Keep only necessary columns for tree features.
    expected_cols = ['tree_id', 'latitude', 'longitude', 'status', 'tree_dbh']
    tree_data = tree_data[expected_cols]

    # Debug: Confirm that columns were filtered.
    print(f"Debug: Filtered tree DataFrame columns: {tree_data.columns.tolist()}")

    # Read training data.
    print("Reading training data...")
    try:
        train_data = pd.read_csv(train_file)
    except FileNotFoundError:
        print(f"Error: Training data file not found at {train_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the training data.
    print(f"Debug: Training DataFrame shape: {train_data.shape}")
    print(f"Debug: Training DataFrame columns: {train_data.columns.tolist()}")

    # Read validation data.
    print("Reading validation data...")
    try:
        validation_data = pd.read_csv(valid_file)
    except FileNotFoundError:
        print(f"Error: Validation data file not found at {valid_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the validation data.
    print(f"Debug: Validation DataFrame shape: {validation_data.shape}")
    print(f"Debug: Validation DataFrame columns: {validation_data.columns.tolist()}")

    # Ensure the training and validation datasets have 'Latitude' and 'Longitude'.
    for df, name in [(train_data, "Training"), (validation_data, "Validation")]:
        if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
            print(f"Error: 'Latitude' and/or 'Longitude' columns not found in the {name} dataset.")
            return

    # Debug: Confirm that latitude and longitude columns are present.
    print("Debug: Latitude and Longitude columns verified in both datasets.")

    # Calculate tree features (counts and average diameters) for training data.
    print("Calculating tree features for training data...")
    train_data = calculate_tree_features_vectorized(train_data.copy(), tree_data, radii)

    # Debug: Print the shape of the training data after feature calculation.
    print(f"Debug: Training DataFrame shape after feature calculation: {train_data.shape}")

    # Calculate tree features for validation data.
    print("Calculating tree features for validation data...")
    validation_data = calculate_tree_features_vectorized(validation_data.copy(), tree_data, radii)

    # Debug: Print the shape of the validation data after feature calculation.
    print(f"Debug: Validation DataFrame shape after feature calculation: {validation_data.shape}")

    # Save the augmented data to new CSV files.
    print("Saving results...")
    train_data.to_csv(output_train_csv, index=False)
    validation_data.to_csv(output_valid_csv, index=False)

    # Debug: Print the final confirmation messages with file paths.
    print(f"Debug: Augmented training data saved to: {output_train_csv}")
    print(f"Debug: Augmented validation data saved to: {output_valid_csv}")

    print("Process completed!")

if __name__ == "__main__":
    main()

Debug: Radii for tree features: [50, 100, 200, 300, 500, 750, 1000]
Reading tree data...
Debug: Tree DataFrame shape: (683788, 45)
Debug: Tree DataFrame columns: ['tree_id', 'block_id', 'created_at', 'tree_dbh', 'stump_diam', 'curb_loc', 'status', 'health', 'spc_latin', 'spc_common', 'steward', 'guards', 'sidewalk', 'user_type', 'problems', 'root_stone', 'root_grate', 'root_other', 'trunk_wire', 'trnk_light', 'trnk_other', 'brch_light', 'brch_shoe', 'brch_other', 'address', 'postcode', 'zip_city', 'community board', 'borocode', 'borough', 'cncldist', 'st_assem', 'st_senate', 'nta', 'nta_name', 'boro_ct', 'state', 'latitude', 'longitude', 'x_sp', 'y_sp', 'council district', 'census tract', 'bin', 'bbl']
Debug: Tree DataFrame shape after filtering for 'Alive' trees: (652173, 45)
Debug: Filtered tree DataFrame columns: ['tree_id', 'latitude', 'longitude', 'status', 'tree_dbh']
Reading training data...
Debug: Training DataFrame shape: (11229, 4)
Debug: Training DataFrame columns: ['Longitu

Processing locations: 100%|██████████| 12/12 [00:09<00:00,  1.26it/s]


Debug: New columns added to locations DataFrame: ['tree_count_50m', 'tree_avg_diam_50m', 'tree_count_100m', 'tree_avg_diam_100m', 'tree_count_200m', 'tree_avg_diam_200m', 'tree_count_300m', 'tree_avg_diam_300m', 'tree_count_500m', 'tree_avg_diam_500m', 'tree_count_750m', 'tree_avg_diam_750m', 'tree_count_1000m', 'tree_avg_diam_1000m']
Debug: Training DataFrame shape after feature calculation: (11229, 18)
Calculating tree features for validation data...
Debug: Locations DataFrame shape: (1040, 3)
Debug: Tree DataFrame shape: (652173, 6)
Debug: Number of locations to process: 1040
Debug: KD-tree for trees created successfully.
Debug: Radii (in meters) for feature computation: [50, 100, 200, 300, 500, 750, 1000]
Debug: Number of valid tree diameters: 652173
Debug: Sample tree diameters: [ 3 21  3 10 21]
Debug: Number of batches to process: 2


Processing locations: 100%|██████████| 2/2 [00:00<00:00,  2.16it/s]


Debug: New columns added to locations DataFrame: ['tree_count_50m', 'tree_avg_diam_50m', 'tree_count_100m', 'tree_avg_diam_100m', 'tree_count_200m', 'tree_avg_diam_200m', 'tree_count_300m', 'tree_avg_diam_300m', 'tree_count_500m', 'tree_avg_diam_500m', 'tree_count_750m', 'tree_avg_diam_750m', 'tree_count_1000m', 'tree_avg_diam_1000m']
Debug: Validation DataFrame shape after feature calculation: (1040, 17)
Saving results...
Debug: Augmented training data saved to: /kaggle/working//training_data_TREE_width_count.csv
Debug: Augmented validation data saved to: /kaggle/working//validation_data_TREE_width_count.csv
Process completed!
